# Data Degradation
Prepare noisy data for the models in the next notebooks.

## Libraries & Drive Access & some constants

In [ ]:

from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets

In [ ]:

from datasets import load_from_disk, load_dataset_builder

In [ ]:
import os
import sys
import time
import random
import shutil
import numpy as np
import torch
import torchvision
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Any
from PIL import Image

# for Data & Image check
import json
import math
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

# Paths. On Google Colab, mount Drive and use the project folder there; running
# locally instead, fall back to the repository root. Either way the structure is
# the same: BASE_DIR / "gan_output" / {weights, samples, metrics_data, reconstruction_examples}.
try:
    from google.colab import drive
    !pip install astra-toolbox
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Computational Imaging")
except ModuleNotFoundError:
    _cwd = Path.cwd().resolve()
    BASE_DIR = next((c for c in (_cwd, _cwd.parent) if (c / "Homeworks").exists()), Path("..").resolve())
sys.path.append(str(BASE_DIR))

from IPPy.IPPy.operators import Blurring
import IPPy.IPPy.utilities as IPPy_utils

# Constants
DEVICE = IPPy_utils.get_device()
BATCHED = True
BATCH_SIZE = 128
INDICES = True

## Basic functions and configuration
Configurations of parameters like noise, type of blur, dataset and samples per class

In [ ]:
@dataclass(frozen=True)
class Config:
  seed: int = 42
  dataset_name: str = "benjamin-paine/imagenet-1k-256x256"
  target_classes: List[int] = (0, 10, 100, 200) # default_factory with lambda to have each istance of Config it's own target_classes
  samples_per_class: int = 1300 #max

  drive_output_dir: Path = BASE_DIR / "dataset_64x64"

  # Forward operator parameters
  img_shape: Tuple[int, int, int] = (3, 64, 64) # img type, (C,H,W)
  blur_kernel_type: str = "gaussian"
  blur_kernel_size: int = 9
  blur_sigma: float = 2.0
  motion_angle: float = 45.0 # unused rn

  noise_levels: List[float] = (0.005, 0.01, 0.05, 0.1)
  device: str = DEVICE

def set_seed(seed: int = 42) -> None:
  """ Set seed for reproducibility """
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  torch.use_deterministic_algorithms(True, warn_only=True)

# Image transformation
pil_to_tensor = torchvision.transforms.ToTensor()
tensor_to_pil = torchvision.transforms.ToPILImage()

## Forward Model

In [ ]:
def forward_model( image_batch: dict[str, list[Any]],
                  indices: list[int], K: Blurring, noise_levels: list[float],
                   device: str, base_seed: int) -> dict[str, Any]:
  """
  Apply  y = K(x) + e

  Parameters:
    image_batch: series of BATCH_SIZE images from the dataset, in format { "image": [img1, ...], "label": [0, ...] }
    indices: list of indices of the images from the dataset, for assuring determinism later on in the function. List as image_batch are multiple images
    K: blurring operator
    noise_levels: list of config.noise_levels
    device: used for GPU or apple-cpu optimization, if available
    base_seed: assuring that different images(and level of noise) have different type of noise, but still deterministic
  """
  batch_images = image_batch.get("image", [])
  batch_size = len(batch_images)

  # in case of empty batches
  # if batch_size == 0:
  #     output_dict = {f"y_{int(round(sigma_n * 1000)):03d}": [] for sigma_n in noise_levels}
  #     output_dict["x"] = []
  #     return output_dict

  # convert into RGB in case it's not
  tensor_list = []
  for img in batch_images:
    if img.mode != "RGB":
      img = img.convert("RGB")
    tensor_list.append(pil_to_tensor(img))

  # blur
  x_batch = torch.stack(tensor_list).to(device).float()
  y_clean = K(x_batch)

  # noise
  output_dict = {f"y_{int(round(sigma_n * 1000)):03d}": [] for sigma_n in noise_levels}
  output_dict["x"] = batch_images

  for i, idx in enumerate(indices):
    img_clean = y_clean[i]

    for sigma_n in noise_levels:
      # specific seed for each image, should be enough to not have collisions
      torch.manual_seed(base_seed + idx + int(sigma_n * 1000))

      y_noisy = img_clean + IPPy_utils.gaussian_noise(img_clean, sigma_n)

      y_noisy_uint8 = (y_noisy * 255.0).clamp(0, 255).to(torch.uint8)
      y_noisy_pil = tensor_to_pil(y_noisy_uint8.cpu())

      key = f"y_{int(round(sigma_n * 1000)):03d}"
      output_dict[key].append(y_noisy_pil)

  return output_dict

## Dataset loading and preprocessing

In [ ]:
config = Config()
set_seed(config.seed)

# definition of the blur operator
blur = Blurring(
  img_shape=config.img_shape,
  kernel_type=config.blur_kernel_type,
  kernel_size=config.blur_kernel_size,
  kernel_variance=config.blur_sigma**2
)

# reading train part of the dataset, collecting samples until config.samples_per_class is satisfied
iterable_ds = load_dataset(config.dataset_name, split="train", streaming=True)
class_datasets = []
for cls in config.target_classes:
  cls_iter = iterable_ds.filter(lambda x, c=cls: x["label"] == c)
  cls_ds = Dataset.from_generator(
      lambda s=cls_iter.take(config.samples_per_class): (yield from s),
      features=iterable_ds.features
  )
  class_datasets.append(cls_ds)
ds = concatenate_datasets(class_datasets)

# --- NUOVO CODICE: DOWNSCALING A 64x64 ---
resize_transform = torchvision.transforms.Resize((64, 64), interpolation=torchvision.transforms.InterpolationMode.BILINEAR, antialias=True)

def downscale_image(example, transform):
    if "image" in example and example["image"] is not None:
        example["image"] = transform(example["image"])
    return example

print("Eseguo il downscaling a 64x64 delle immagini pulite...")
ds = ds.map(
    downscale_image, 
    batched=False, 
    fn_kwargs={"transform": resize_transform},
    num_proc=4,
    desc="Downscaling"
)
# -----------------------------------------

# doing the degradation
print("Applicazione di blur e rumore...")
ds_degraded = ds.map(
  forward_model,
  batched=BATCHED,
  batch_size=BATCH_SIZE,
  with_indices=INDICES,
  remove_columns=["image"],
  fn_kwargs={
    "K": blur,
    "noise_levels": config.noise_levels,
    "device": config.device,
    "base_seed": config.seed
  }
)

## Data separation & saving

In [ ]:
# splitting the dataset into train 80% - validation 10% - testing 10%
train_test_split = ds_degraded.train_test_split(test_size=0.2, seed=config.seed, stratify_by_column="label")
val_test_split = train_test_split['test'].train_test_split(test_size=0.5, seed=config.seed, stratify_by_column="label")

final_dataset = DatasetDict({
  'train': train_test_split['train'],
  'validation': val_test_split['train'],
  'test': val_test_split['test']
})

# saving into drive
config.drive_output_dir.mkdir(parents=True, exist_ok=True)
final_dataset.save_to_disk(str(config.drive_output_dir))

# saving the class names
builder = load_dataset_builder(config.dataset_name)
class_names = builder.info.features["label"].names
id2label = {cls: class_names[cls] for cls in config.target_classes}
with open(config.drive_output_dir / "labels.json", "w") as f:
    json.dump(id2label, f, indent=2)

## Data & Image check

In [ ]:
try:
  # if this is executed right after the cells above, i do not need to re-define variables
  config

except NameError:
  # if "Dataset loading and preprocessing" and "Data separation & Saving" cells are not executed
  config = Config()
  set_seed(config.seed)
  blur = Blurring(
    img_shape=config.img_shape,
    kernel_type=config.blur_kernel_type,
    kernel_size=config.blur_kernel_size,
    kernel_variance=config.blur_sigma**2
  )
  final_dataset = load_from_disk(str(config.drive_output_dir))
  print(f"Dataset caricato da: {config.drive_output_dir}")

  # === NUOVO: Ricreiamo ds_degraded ===
  # Per mantenere lo stato dell'ambiente perfettamente coerente, uniamo i vari
  # split di final_dataset per ricreare la variabile ds_degraded
  from datasets import concatenate_datasets
  ds_degraded = concatenate_datasets([
      final_dataset['train'],
      final_dataset['validation'],
      final_dataset['test']
  ])
# === NUOVO: Garantiamo l'esistenza di builder e class_names ===
# Leggiamo i metadata del dataset così queste variabili esistono sempre nell'ambiente
builder = load_dataset_builder(config.dataset_name)
class_names = builder.info.features["label"].names
# for label of the class used
if (config.drive_output_dir / "labels.json").exists():
  # reading from the file on drive
  with open(config.drive_output_dir / "labels.json") as f:
    id2label = json.load(f)
    id2label = {int(k): v for k, v in id2label.items()}
else:
  # reading from the official dataset
  id2label = {cls: class_names[cls] for cls in config.target_classes}
print("=== Split sizes ===")
for split, ds_split in final_dataset.items():
  print(f"  {split:>10}: {len(ds_split):>5} samples")
total = sum(len(v) for v in final_dataset.values())
print(f"  {'TOTAL':>10}: {total:>5} samples")
print("\n=== Label distribution ===")
for split, ds_split in final_dataset.items():
  counts = Counter(ds_split['label'])
  print(f"  {split}: { {k: counts[k] for k in sorted(counts)} }")
print("\n=== Columns ===")
print(" ", final_dataset['train'].column_names)
print("\n=== Pixel value ranges (first 5 train samples) ===")
sample = final_dataset['train'].select(range(5))
for col in ['x'] + [f"y_{int(round(sigma_n * 1000)):03d}" for sigma_n in config.noise_levels]:
  vals = [np.array(img) for img in sample[col]]
  arr = np.stack(vals)
  print(f"  {col}: min={arr.min()}, max={arr.max()}, dtype={arr.dtype}")
#  verify ||e|| / ||Kx|| ≈ sigma_n for each level of noise
print("\n=== Noise level verification (mean over 20 samples) ===")
sample = final_dataset['train'].select(range(20))
x_tensors   = torch.stack([pil_to_tensor(img) for img in sample['x']]).float().to(config.device)
y_clean_est = blur(x_tensors)  # Kx
for sigma_n in config.noise_levels:
  key = f"y_{int(round(sigma_n * 1000)):03d}"
  y_tensors = torch.stack([pil_to_tensor(img) for img in sample[key]]).float().to(config.device)
  noise     = y_tensors - y_clean_est
  measured_sigma = noise.std(dim=(1, 2, 3))
  print(f"  {key}: expected={sigma_n:.3f}  measured={measured_sigma.mean():.4f} ± {measured_sigma.std():.4f}")
print("\n=== Blur MSE ===")
mse = ((x_tensors - y_clean_est) ** 2).mean()
print("Blur MSE:", mse.item())
print("\n=== Image Shape ===")
sample = final_dataset["train"][0]
for key in ["x"] + [
    f"y_{int(round(s*1000)):03d}"
    for s in config.noise_levels
]:
    print(key, np.array(sample[key]).shape)
# === Plotting ===
cols_to_show = ['x'] + [f"y_{int(round(s*1000)):03d}" for s in config.noise_levels]
titles       = ['x (clean)', *[f"y σ={s}" for s in config.noise_levels]]
n_per_class  = 2
n_rows = len(config.target_classes) * n_per_class
n_cols = len(cols_to_show)
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(n_cols * 3, n_rows * 3)
)
labels_array = np.array(final_dataset['train']['label'])
row = 0
for cls in config.target_classes:
    all_indices = np.where(labels_array == cls)[0][:n_per_class].tolist()
    for idx in all_indices:
        sample_row = final_dataset['train'][int(idx)]
        for col_i, col in enumerate(cols_to_show):
            ax = axes[row][col_i]
            ax.imshow(sample_row[col])
            ax.axis('off')
            if col_i == 0:
                ax.set_ylabel(f"{id2label[cls]}", fontsize=9, rotation=90, labelpad=4)
            if row == 0:
                ax.set_title(titles[col_i], fontsize=9)
        row += 1
plt.suptitle("Degradation check: x vs y per noise level", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(str(config.drive_output_dir / "image_check.png"), dpi=100, bbox_inches='tight')
plt.show()